In [ ]:
from openai import OpenAI
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
import huggingface_hub
import os

c:\Users\kwokd\anaconda3\envs\tennis_analysis\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load HuggingFace LLM

In [ ]:
model_name = "meta-llama/Llama-3.2-3B-Instruct"
hf_token = os.environ('HF_TOKEN_KEY')

huggingface_hub.login(hf_token)

In [3]:
torch.random.manual_seed(0)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="cuda",
    torch_dtype="auto",
    trust_remote_code=True
)

c:\Users\kwokd\anaconda3\envs\tennis_analysis\lib\site-packages\huggingface_hub\file_download.py:144: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\kwokd\.cache\huggingface\hub\models--meta-llama--Llama-3.2-3B-Instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading checkpoint shards: 100%|██████████| 2/2 [00:02<00:00,  1.35s/it]


In [4]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token_id = tokenizer.eos_token_id

In [6]:
pipe = pipeline("text-generation",
                model=model,
                tokenizer=tokenizer)

Device set to use cuda


In [7]:
def get_response_hf(pipe, prompt, temperature=0):
    messages = [
        {"role" : "user",
         "content" : prompt}
    ]

    sampling = False if temperature == 0 else True 

    generation_args = {
        "max_new_tokens" : 2000,
        "return_full_text" : False,
        "temperature" : temperature,
        "do_sample" : sampling
    }

    output = pipe(messages, **generation_args)
    return output[0]["generated_text"]

In [8]:
output = get_response_hf(pipe, "What is the capital of Singapore?")
print(output)

c:\Users\kwokd\anaconda3\envs\tennis_analysis\lib\site-packages\transformers\generation\configuration_utils.py:631: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
c:\Users\kwokd\anaconda3\envs\tennis_analysis\lib\site-packages\transformers\generation\configuration_utils.py:636: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(


The capital of Singapore is Singapore itself, as it is a city-state.


## Prompt Engineering

### Principle 1 -- Write clear and Specific Instructions

- Use Delimiters to clearly indicate parts of the prompt

In [12]:
review = "This product is awesome and I like it so much"

# Clear Distinction between the instructions and the actual review
prompt = f"""
        Determine the sentiment (Positive, Negative, Neutral) of the following review. Please provide a simple analysis on why is positive/negative/netural.
        The review is between three backticks. Provide the analysis in between the dashes below.

        ```{review}```
        ---(analysis)---
"""

output = get_response_hf(pipe, prompt)
print(output)

c:\Users\kwokd\anaconda3\envs\tennis_analysis\lib\site-packages\transformers\generation\configuration_utils.py:631: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
c:\Users\kwokd\anaconda3\envs\tennis_analysis\lib\site-packages\transformers\generation\configuration_utils.py:636: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(


Positive

- The review starts with "This product is awesome", which is a strong positive statement.
- The use of the word "awesome" indicates a high level of satisfaction and enthusiasm.
- The phrase "I like it so much" reinforces the positive sentiment, emphasizing the reviewer's strong liking for the product.


#### Ask for a structured output

- Ask for a specific format e.g., JSON, XML, etc

In [14]:
review = "This product is awesome and I like it so much"

# Clear Distinction between the instructions and the actual review
prompt = f"""
        Determine the sentiment (Positive, Negative, Neutral) of the following review. Please provide a simple analysis on why is positive/negative/netural.
        The review is between three backticks. Provide the analysis in between the dashes below.

        ```{review}```
        ---(analysis)---

        Generate the answer in a JSON format that has the following keys:
        - review : user's review
        - sentiment : string that is one of those values (Positive, Negative, Neutral)
        - analysis : analysis on why is it positive/negative/neutral

        Always respond with a valid JSON. DO NOT include any extra characters, symbols, or text outside the JSON itself.
"""

output = get_response_hf(pipe, prompt)
print(output)

c:\Users\kwokd\anaconda3\envs\tennis_analysis\lib\site-packages\transformers\generation\configuration_utils.py:631: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
c:\Users\kwokd\anaconda3\envs\tennis_analysis\lib\site-packages\transformers\generation\configuration_utils.py:636: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(


```
{
  "review": "This product is awesome and I like it so much",
  "sentiment": "Positive",
  "analysis": "The review is positive because it uses strong positive adjectives such as 'awesome' and expresses a strong liking for the product."
}
```


#### Few shot learning

In [17]:
review = "The lighting in the shop is so warm, and it makes the place feel inviting"

prompt = f"""
        Determine the category of the review.
        It is in one of these 4 options: ("Service", "Quality", "Ambience", "Pricing").
        The user review is between the 3 backticks.

        ```
        {review}
        ```

        Here are some examples of reviews and their categories

        Review: "The barista was incredibly friendly and made my drink quickly" 
        Category: Service

        Review: "The cappuccino was perfect, and the beans tasted fresh." 
        Category: Quality

        Review: "I love the cozy seating and relaxing music in the shop"
        Category: Ambience

        Review: "The price are a bit too high compared to other cafes nearby."
        Category: Pricing

        Generate the answer in a JSON format that has the following fields:
        - review : user's review
        - category: one of the values from the options ("Service", "Quality", "Ambience", "Pricing")
        - sentiment: one of the categories ("Positive", "Negative", "Neutral")

        Always respond with a valid JSON. DO NOT include any extra characters, symbols or text outside the JSON itself.        
"""

output = get_response_hf(pipe, prompt)
print(output)

c:\Users\kwokd\anaconda3\envs\tennis_analysis\lib\site-packages\transformers\generation\configuration_utils.py:631: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
c:\Users\kwokd\anaconda3\envs\tennis_analysis\lib\site-packages\transformers\generation\configuration_utils.py:636: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(


```
[
  {
    "review": "The lighting in the shop is so warm, and it makes the place feel inviting",
    "category": "Ambience",
    "sentiment": "Positive"
  }
]
```


## Principle 2 - Give the model time to think

- Specify the steps required to complete the task
- Chain of Thought
    - Make the model think step by step

In [20]:
prompt = """ 
        I bought 2 balls with $10. One ball is more expensive than the other by $1.

        How much does the more expensive ball costs?

        Please provide the answer in a single number
"""

output = get_response_hf(pipe, prompt)
print(output)

Let's say the less expensive ball costs x dollars. Since the more expensive ball costs $1 more, it costs x + $1.

You spent a total of $10 on both balls, so we can set up the equation:

x + (x + $1) = $10

Combine like terms:

2x + $1 = $10

Subtract $1 from both sides:

2x = $9

Divide both sides by 2:

x = $4.50

So, the less expensive ball costs $4.50. The more expensive ball costs $4.50 + $1 = $5.50.

The more expensive ball costs $5.50.


In [21]:
prompt = """ 
        I bought 2 balls with $10. One ball is more expensive than the other by $1.

        How much does the more expensive ball costs?

        Please think about this step by step and provide the answer at the end.
"""

output = get_response_hf(pipe, prompt)
print(output)

c:\Users\kwokd\anaconda3\envs\tennis_analysis\lib\site-packages\transformers\generation\configuration_utils.py:631: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
c:\Users\kwokd\anaconda3\envs\tennis_analysis\lib\site-packages\transformers\generation\configuration_utils.py:636: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(


To solve this problem, let's break it down step by step.

1. We have 2 balls, and one is more expensive than the other by $1.
2. Let's assume the price of the less expensive ball is x dollars.
3. Since the more expensive ball is $1 more than the less expensive ball, its price is x + $1.
4. We know that the total amount we spent is $10.
5. We can set up an equation to represent the total cost: x + (x + $1) = $10.
6. Combine like terms: 2x + $1 = $10.
7. Subtract $1 from both sides: 2x = $9.
8. Divide both sides by 2: x = $4.50.
9. Since the more expensive ball costs $1 more than the less expensive ball, its price is x + $1 = $4.50 + $1 = $5.50.

Therefore, the more expensive ball costs $5.50.


#### Specify the steps required to complete the task

In [22]:
prompt = """
        How many r's are in Strawberry?
"""

output = get_response_hf(pipe, prompt)
print(output)

c:\Users\kwokd\anaconda3\envs\tennis_analysis\lib\site-packages\transformers\generation\configuration_utils.py:631: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
c:\Users\kwokd\anaconda3\envs\tennis_analysis\lib\site-packages\transformers\generation\configuration_utils.py:636: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(


There are 2 R's in the word "Strawberry".


In [23]:
prompt = """ 
        How many r's are in strawberry?

        Follow the steps bellow to count the r's in Strawberry:
        1. Break down the word into letters
        2. For each letter write 1 if it's an r and 0 if it isn't
        3. Now have a counter that counts the number of 1s
        4. Write down the final answer as follows - Total number of r: answer
"""

output = get_response_hf(pipe, prompt)
print(output)

c:\Users\kwokd\anaconda3\envs\tennis_analysis\lib\site-packages\transformers\generation\configuration_utils.py:631: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
c:\Users\kwokd\anaconda3\envs\tennis_analysis\lib\site-packages\transformers\generation\configuration_utils.py:636: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(


To count the number of r's in the word "strawberry", let's follow the steps:

1. Break down the word into letters:
   s-t-r-a-w-b-e-r-r-y

2. For each letter, write 1 if it's an r and 0 if it isn't:
   s - 0
   t - 0
   r - 1
   a - 0
   w - 0
   b - 0
   e - 0
   r - 1
   r - 1
   y - 0

3. Now, have a counter that counts the number of 1s:
   Counter: 1 + 1 + 0 = 2

4. Write down the final answer:
   Total number of r: 2


## Text Summarisation

In [24]:
customer_review = """
        I recently visited Coffee Haven and ordered a caramel latte with almond milk. \
        The staff was friendly, but the service was slow. It took almost 20 minutes to get my drink. \
        The latte itself was too sweet for my liking, and I could barely taste the coffee. \
        However, the ambiance was cozy, and I loved the music they played. \
        I might come back for the atmosphere, but not for the coffee.
"""

prompt = f"""
        The user review is between xml tag called customer_review 

        <customer_review>
        {customer_review}
        </customer_review>

        Please summarise this review in one sentence
"""

output = get_response_hf(pipe, prompt)
print(output)

c:\Users\kwokd\anaconda3\envs\tennis_analysis\lib\site-packages\transformers\generation\configuration_utils.py:631: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
c:\Users\kwokd\anaconda3\envs\tennis_analysis\lib\site-packages\transformers\generation\configuration_utils.py:636: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(


The reviewer had a mixed experience at Coffee Haven, enjoying the cozy atmosphere and music, but was disappointed with the slow service and overly sweet latte.


## Translation

In [25]:
prompt = f"""
        The user review is between xml tag called customer_review 

        <customer_review>
        {customer_review}
        </customer_review>

        Please translate the customer review to french and provide only the translated text without any XML tags.
"""

output = get_response_hf(pipe, prompt)
print(output)

c:\Users\kwokd\anaconda3\envs\tennis_analysis\lib\site-packages\transformers\generation\configuration_utils.py:631: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
c:\Users\kwokd\anaconda3\envs\tennis_analysis\lib\site-packages\transformers\generation\configuration_utils.py:636: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(


Voici le résumé du commentaire en français :

J'ai récemment visité le Café Haven et j'ai commandé un latte au caramel avec de la lait d'amande. Le personnel était amical, mais le service était lent. Il a fallu presque 20 minutes pour obtenir mon verre. Le latte lui-même était trop sucré pour mon goût, et je n'ai presque pas pu goûter le café. Cependant, l'ambiance était chaleureuse, et j'adore la musique qu'ils jouent. Je pourrais revenir pour l'atmosphère, mais pas pour le café.


## NER - Named Entity Recognition

In [26]:
prompt = f"""
The user review is between xml tag called customer_review

<customer_review>
{customer_review}
</customer_review>

Refer to the list of named entities below:
Person Names
Organizations
Locations
Cities
Countries
Continents
Regions
Dates
Times
Monetary Values
Percentages
Quantities
Products
Events

Please extract all named entities and their types.
Use the following format.
named entity: type
"""

output = get_response_hf(pipe, prompt)
print(output)

c:\Users\kwokd\anaconda3\envs\tennis_analysis\lib\site-packages\transformers\generation\configuration_utils.py:631: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
c:\Users\kwokd\anaconda3\envs\tennis_analysis\lib\site-packages\transformers\generation\configuration_utils.py:636: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(


Here are the named entities extracted from the user review:

1. Coffee Haven: Organization
2. Caramel latte: Product
3. Almond milk: Product
4. Coffee: Product
5. Music: Product (Note: Music is not typically considered a product, but in this context, it's being referred to as a product, so I've included it)
6. Atmosphere: Location (Note: Atmosphere is not a physical location, but in this context, it's being referred to as a location, so I've included it)

There are no person names, locations, cities, countries, continents, regions, dates, times, monetary values, percentages, or quantities mentioned in the review.
